# nano-relation-extractor: train, quantise, package

This notebook drives the same stages as `uv run nano-re all`, one cell at a
time, so intermediate results can be inspected. All logic lives in the
`nano_re` package; these cells only orchestrate it.

Everything the pipeline produces stays on this machine. Nothing is uploaded
anywhere, and no credentials are needed: the Hugging Face Hub is used only to
download the public DocRED corpus and the public pretrained encoder.

## Section 1: environment

Run these once in a terminal, not in the notebook:

```bash
uv init --package --name nano-re --python 3.11
uv venv
uv add torch transformers datasets accelerate onnx onnxruntime onnxscript optimum huggingface_hub jupyterlab scikit-learn evaluate seqeval
uv run jupyter lab
```

In [ ]:
from dataclasses import replace

from nano_re.config import PipelineConfig
from nano_re.pipeline import Pipeline

config = PipelineConfig.from_env()
config = config.with_overrides(
    data=replace(config.data, limit=None),
    training=replace(config.training, epochs=3),
)
pipeline = Pipeline(config)

print("Backbone:", config.model.backbone_name)
print("Dataset:", config.data.dataset_repo_id)
print("Train split:", config.data.train_split)
print("Artifacts:", config.artifacts_dir.resolve())

## Section 2: data pipeline

Downloads DocRED, derives the label schema and encodes the evaluation split.
No credentials are involved: both the corpus and the encoder are public.

In [ ]:
schema = pipeline.prepare()

print("BIO tags:", schema.bio_labels)
print("Relations:", schema.num_relation_labels)
print("Example:", "P17", "->", schema.describe_relation("P17"))

In [ ]:
documents = pipeline.data_module.load_documents(config.data.eval_split)
encoded = pipeline.data_module.encoder.encode(documents[0])
document = documents[0]

print("Document:", document.doc_id)
print("Words:", document.num_words, "| entities:", document.num_entities)
print("Sub-words:", encoded.input_ids.shape[0], "| candidate pairs:", encoded.num_pairs)
print("Mention mask rows sum to one:", encoded.mention_mask.sum(-1)[:5].tolist())

for triple in document.relations[:3]:
    head = document.entities[triple.head].name
    tail = document.entities[triple.tail].name
    print(f"  {head} --[{schema.describe_relation(triple.relation)}]--> {tail}")

## Section 3: multi-task training

Trains the shared encoder with both heads under
`L = alpha * L_NER + beta * L_RE`, with mixed precision on CUDA, gradient
clipping and per-epoch evaluation. The trained model, its tokenizer and the
training report are written to the artifact directory.

In [ ]:
training_report = pipeline.train()

best = training_report.best_evaluation
print(f"Best epoch: {training_report.best_epoch}")
print(f"NER micro F1:      {best.ner.f1:.4f}")
print(f"Relation micro F1: {best.relation.f1:.4f}")
print(f"Relation recall ceiling: {best.relation_recall_ceiling:.4f}")

## Section 4: ONNX export and INT8 quantisation

The exporter verifies the graph against PyTorch twice: once on the traced batch
and once with different batch, sequence, entity and pair counts. The second
check is what proves the dynamic axes actually hold.

In [ ]:
artifacts = pipeline.export()

print("Exporter:", artifacts.export.exporter, "| opset:", artifacts.export.opset_version)
print("Dynamic shapes verified:", artifacts.export.dynamic_shapes_verified)
print(f"Max deviation: {artifacts.export.max_relation_deviation:.2e}")
print(f"Size: {artifacts.quantization.source_bytes / 1e6:.1f} MB -> "
      f"{artifacts.quantization.target_bytes / 1e6:.1f} MB "
      f"({artifacts.quantization.compression_ratio:.2f}x)")

In [ ]:
benchmark = pipeline.benchmark(measure_accuracy=True)

print(f"FP32: {benchmark.fp32.median_ms:6.2f} ms/page  p95 {benchmark.fp32.p95_ms:6.2f}  "
      f"{benchmark.fp32.size_mb:7.1f} MB")
print(f"INT8: {benchmark.int8.median_ms:6.2f} ms/page  p95 {benchmark.int8.p95_ms:6.2f}  "
      f"{benchmark.int8.size_mb:7.1f} MB")
print(f"Speedup: {benchmark.speedup:.2f}x | size reduction: {benchmark.size_reduction:.1%}")
print(f"NER F1 delta: {benchmark.ner_f1_delta:+.4f} | "
      f"relation F1 delta: {benchmark.relation_f1_delta:+.4f}")

## Section 5: local bundle

The model card is rendered from the reports above, so every number in it comes
from a measurement rather than from prose.

The packaging stage writes `MODEL_CARD.md`, inventories the artifact directory,
verifies that nothing expected is missing and writes `MANIFEST.json`. The
result is a directory you can copy anywhere and run without this package.

In [ ]:
bundle = pipeline.package(
    training=training_report,
    benchmark=benchmark,
    quantization=artifacts.quantization,
)
print(bundle.render())
print()
print("Complete:", bundle.is_complete)

In [ ]:
print((pipeline.artifacts_dir / "MODEL_CARD.md").read_text(encoding="utf-8"))